<a href="https://colab.research.google.com/github/cyanm002-maker/Cascade/blob/main/Cascade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Libraries

In [2]:
import os
import shutil
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score


# This part is for the pytorch and cuda

In [3]:
from torch.cuda import is_available
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print("Pytorch version:",torch.__version__)
print("CUDA available:",torch.cuda.is_available())
if torch.cuda.is_available():
  print("GPU:",torch.cuda.get_device_name(0))

Pytorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


# Inspect the dataset

In [6]:
from google.colab import drive
drive.mount('/content/drive')

src_path = "/content/drive/MyDrive/Cascade/dataset_2_1 (1).hdf5"
dst_path = "/content/dataset_2_1.hdf5"
shutil.copy(src_path, dst_path)

if not os.path.exists(dst_path):
  shutil.copy(src_path, dst_path)

with h5py.File(dst_path, 'r') as f:
  print("HDF5 keys:", list(f.keys()))
  for k in f.keys():
      print(f"{k}: shape={f[k].shape}, dtype={f[k].dtype}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HDF5 keys: ['incident_energies', 'showers']
incident_energies: shape=(100000, 1), dtype=float64
showers: shape=(100000, 6480), dtype=float64


# Transform,DataSet and batch check

In [9]:
def encode_energy(energy_gev):
  return np.log10(np.clip(energy_gev, 1e-10, None))


def preprocess_voxels(voxel_mev, eps=1e-6):
  clean = np.clip(voxel_mev.astype(np.float32), 0.0, None)
  return np.log(clean + eps)

def invert_voxels(pred_log, eps=1e-6, threshold_mev = 0.015):
  x = np.exp(pred_log) - eps
  x = np.clip(x, 0.0, None)
  return np.where(x > threshold_mev, x, 0.0)

class ShowerForgeDataset(Dataset):
  def __init__(self, hdf5_path, shower_key, energy_key, max_events=None, eps=1e-6):
    self.hdf5_path = hdf5_path
    self.shower_key = shower_key
    self.energy_key = energy_key
    self.eps = eps
    self.max_events = max_events
    with h5py.File(self.hdf5_path, 'r') as f:
      total = f[self.energy_key].shape[0]
      self.length = min(total, max_events) if max_events is not None else total
      self.energies = f[self.energy_key][: self.length].astype(np.float32).squeeze()

  def __len__(self):
    return self.length

  def __getitem__(self, index):
    with h5py.File(self.hdf5_path, "r") as f:
        voxel = f[self.shower_key][index].astype(np.float32)
    x_log = preprocess_voxels(voxel, eps=self.eps)
    c_log = encode_energy(self.energies[index])
    return torch.from_numpy(x_log).float(), torch.tensor([c_log], dtype=torch.float32)

# testing on 5k shower
ds = ShowerForgeDataset("/content/dataset_2_1.hdf5", "showers", "incident_energies", max_events=5_000)
loader = DataLoader(ds, batch_size=64, shuffle=True)

x_batch, c_batch = next(iter(loader))
print("x_batch:", x_batch.shape, "c_batch:", c_batch.shape)

x_batch: torch.Size([64, 6480]) c_batch: torch.Size([64, 1])
